# Antonacci Global Equities Momentum (GEM)

The textbook retail TAA strategy (Gary Antonacci, *Dual Momentum Investing*). Monthly:

1. **Absolute momentum.** Is SPY's 12-month total return greater than T-bills (BIL)?
2. If yes, hold whichever of **SPY** or **VEU** had the higher 12-month total return (relative momentum).
3. If no, sit in **AGG** (US aggregate bonds).

One asset held at a time, monthly rebalance, no leverage. About as retail-friendly as it gets.

In [ ]:
import pandas as pd

from alpha_lab.backtest.metrics import summary
from alpha_lab.backtest.vector import run_backtest
from alpha_lab.data.loaders.yfinance import load_prices
from alpha_lab.portfolio.long_only import fixed_weight_returns
from alpha_lab.reporting.charts import drawdown_chart, equity_curve, heatmap_monthly

RISK_ON = ["SPY", "VEU"]      # US equities vs ex-US equities
CASH = "BIL"                   # 1-3 month T-bills
DEFENSIVE = "AGG"              # US aggregate bonds
ALL = RISK_ON + [CASH, DEFENSIVE]

START = "2008-01-01"           # BIL inception May 2007; allow burn-in
END = None
LOOKBACK_M = 12                 # months
REBAL = "ME"
COSTS_BPS = 1.0
SLIPPAGE_BPS = 2.0

In [ ]:
prices = load_prices(ALL, start=START, end=END).dropna(how="any")
prices.tail()

In [ ]:
# Month-end snapshots, then 12m total return per asset.
monthly = prices.resample("ME").last()
ret_12m = monthly / monthly.shift(LOOKBACK_M) - 1

# Relative momentum: SPY vs VEU.
pick_risk = ret_12m[RISK_ON].idxmax(axis=1)
# Absolute momentum filter: equity winner must beat cash.
risk_on = ret_12m[RISK_ON].max(axis=1) > ret_12m[CASH]

# Build target weights: 100% in chosen asset.
w = pd.DataFrame(0.0, index=monthly.index, columns=ALL)
for date, asset in pick_risk.items():
    if pd.isna(asset):
        continue
    if bool(risk_on.loc[date]):
        w.loc[date, asset] = 1.0
    else:
        w.loc[date, DEFENSIVE] = 1.0

# Expand to daily; run_backtest applies the 1-period lag.
weights = w.reindex(prices.index).ffill().fillna(0.0)
weights.tail()

In [ ]:
# What fraction of the time is each asset held?
(w.replace(0.0, pd.NA).notna().sum() / w.notna().sum().sum()).rename("share_of_months")

In [ ]:
res = run_backtest(weights, prices, rebalance=REBAL, costs_bps=COSTS_BPS, slippage_bps=SLIPPAGE_BPS)

rets = prices.pct_change().fillna(0.0)
bench = pd.DataFrame({
    "GEM": res.returns,
    "BH SPY": rets["SPY"],
    "60/40 (SPY/AGG)": fixed_weight_returns(rets, {"SPY": 0.6, "AGG": 0.4}),
}).dropna()
pd.DataFrame({name: pd.Series(summary(s)) for name, s in bench.items()})

In [ ]:
annual_turnover = res.turnover.resample("YE").sum()
annual_costs = res.costs.resample("YE").sum()
pd.DataFrame({"turnover_per_year": annual_turnover, "cost_drag_per_year": annual_costs})

In [ ]:
equity_curve(bench)

In [ ]:
drawdown_chart(res.returns)

In [ ]:
heatmap_monthly(res.returns)

## Read

- The original Antonacci backtest used Russell 1000 / ACWI ex-US / Barclays Agg / T-bills going back decades; our ETF proxies only have ~17 years of clean overlap. Expect lower edge than the book.
- The biggest visible 'wins' for GEM are 2008 (out of equities into AGG) and 2022 (briefly defensive). Whether it works going forward depends on whether bonds keep playing defense.
- All-in implementation: one brokerage account, three ETFs, one trade decision per month.